# GPT-3.5 Turbo Fine-tuning for Customer Churn Prediction

This notebook demonstrates how to fine-tune GPT-3.5 Turbo using OpenAI's Fine-tuning API for customer churn prediction.

## Prerequisites
- OpenAI API key with fine-tuning access
- Training dataset in JSONL format (generated from `generate_customer_churn.py`)
- `openai` Python package installed


## 1. Installation and Setup


In [ ]:
# Install OpenAI package if not already installed
!pip install openai


In [ ]:
import os
import json
import time
from datetime import datetime
from openai import OpenAI

In [ ]:
# Load API key from shell environment variable
# If the automatic detection doesn't work, you can set it directly:
# os.environ['OPEN_AI_FINE_TUNING_KEY'] = 'your-api-key-here'


from IPython import get_ipython
import os

api_key = get_ipython().getoutput('echo $OPEN_AI_FINE_TUNING_KEY')
if api_key and api_key[0].strip():
    os.environ['OPEN_AI_FINE_TUNING_KEY'] = api_key[0].strip()
    print("✅ API key loaded from shell")
else:
    raise ValueError("API key not found")


In [ ]:
client = OpenAI(
    api_key=os.getenv('OPEN_AI_FINE_TUNING_KEY')  # Reads from environment variable
)

# Verify API key is set
if not client.api_key:
    raise ValueError("Please set your OPENAI_API_KEY environment variable or update the client initialization above")

print("✅ OpenAI client initialized successfully")


In [ ]:
# Specify your training file path
TRAINING_FILE = "churn_dataset_gpt35_turbo.jsonl"  # Update this to your file path

# Verify file exists
if not os.path.exists(TRAINING_FILE):
    raise FileNotFoundError(f"Training file not found: {TRAINING_FILE}")

# Check file format (first few lines)
print("📄 Checking training file format...")
with open(TRAINING_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 2:  # Show first 2 examples
            break
        record = json.loads(line)
        print(f"\nExample {i+1}:")
        print(json.dumps(record, indent=2))

print(f"\n✅ Training file ready: {TRAINING_FILE}")


In [ ]:
# Upload the training file to OpenAI
print(f"📤 Uploading {TRAINING_FILE} to OpenAI...")

with open(TRAINING_FILE, 'rb') as f:
    training_file = client.files.create(
        file=f,
        purpose='fine-tune'
    )

print(f"✅ File uploaded successfully!")
print(f"   File ID: {training_file.id}")
print(f"   File Name: {training_file.filename}")
print(f"   File Size: {training_file.bytes} bytes")
print(f"   Status: {training_file.status}")

# Save file ID for later use
TRAINING_FILE_ID = training_file.id


## 4. Create Fine-tuning Job


In [ ]:
# Create a fine-tuning job
print("🚀 Creating fine-tuning job...")

fine_tuning_job = client.fine_tuning.jobs.create(
    training_file=TRAINING_FILE_ID,
    model="gpt-3.5-turbo",  # Base model
    hyperparameters={
        "n_epochs": 3  # Number of epochs (OpenAI recommends 1-4, default is auto)
    },
    suffix="churn-prediction"  # Optional: adds suffix to model name
)

print(f"✅ Fine-tuning job created!")
print(f"   Job ID: {fine_tuning_job.id}")
print(f"   Status: {fine_tuning_job.status}")
print(f"   Model: {fine_tuning_job.model}")
print(f"   Created At: {datetime.fromtimestamp(fine_tuning_job.created_at)}")

# Save job ID for monitoring
FINE_TUNING_JOB_ID = fine_tuning_job.id


## 5. Monitor Fine-tuning Progress


In [ ]:
# Function to check fine-tuning job status
def check_fine_tuning_status(job_id):
    """Check the status of a fine-tuning job"""
    job = client.fine_tuning.jobs.retrieve(job_id)
    return job

# Monitor the job (run this cell multiple times to check progress)
job = check_fine_tuning_status(FINE_TUNING_JOB_ID)

print(f"📊 Fine-tuning Job Status:")
print(f"   Job ID: {job.id}")
print(f"   Status: {job.status}")
print(f"   Model: {job.model}")
print(f"   Created At: {datetime.fromtimestamp(job.created_at)}")

if hasattr(job, 'finished_at') and job.finished_at:
    print(f"   Finished At: {datetime.fromtimestamp(job.finished_at)}")
    duration = job.finished_at - job.created_at
    print(f"   Duration: {duration/60:.1f} minutes")

if hasattr(job, 'trained_tokens') and job.trained_tokens is not None:
    print(f"   Trained Tokens: {job.trained_tokens:,}")

if hasattr(job, 'error'):
    print(f"   Error: {job.error}")

if job.status == 'succeeded':
    print(f"\n✅ Fine-tuning completed successfully!")
    print(f"   Fine-tuned Model: {job.fine_tuned_model}")
    FINE_TUNED_MODEL = job.fine_tuned_model
elif job.status == 'failed':
    print(f"\n❌ Fine-tuning failed!")
    if hasattr(job, 'error'):
        print(f"   Error: {job.error}")
elif job.status in ['validating_files', 'queued', 'running']:
    print(f"\n⏳ Fine-tuning in progress... (Status: {job.status})")
    print(f"   This may take several minutes to hours depending on dataset size.")
    print(f"   Run this cell again to check the latest status.")


In [ ]:
# Optional: Continuous monitoring (uncomment to use)
# This will poll the job status every 30 seconds until completion

# print("🔄 Monitoring fine-tuning job (press Ctrl+C to stop)...")
# try:
#     while True:
#         job = check_fine_tuning_status(FINE_TUNING_JOB_ID)
#         status = job.status
#         
#         print(f"\r[{datetime.now().strftime('%H:%M:%S')}] Status: {status}", end='', flush=True)
#         
#         if status == 'succeeded':
#             print(f"\n\n✅ Fine-tuning completed!")
#             print(f"   Fine-tuned Model: {job.fine_tuned_model}")
#             FINE_TUNED_MODEL = job.fine_tuned_model
#             break
#         elif status == 'failed':
#             print(f"\n\n❌ Fine-tuning failed!")
#             if hasattr(job, 'error'):
#                 print(f"   Error: {job.error}")
#             break
#         
#         time.sleep(30)  # Check every 30 seconds
# except KeyboardInterrupt:
#     print("\n\n⏸ Monitoring stopped by user")


## 6. Retrieve Fine-tuned Model Name


In [ ]:
# Once training is complete, get the fine-tuned model name
job = check_fine_tuning_status(FINE_TUNING_JOB_ID)

if job.status == 'succeeded':
    FINE_TUNED_MODEL = job.fine_tuned_model
    print(f"✅ Fine-tuned model ready!")
    print(f"   Model Name: {FINE_TUNED_MODEL}")
    print(f"\n💾 Save this model name for inference:")
    print(f"   FINE_TUNED_MODEL = '{FINE_TUNED_MODEL}'")
else:
    print(f"⏳ Training not yet complete. Current status: {job.status}")
    print(f"   Please wait and check again.")


## 7. Test the Fine-tuned Model


In [ ]:
# Set your fine-tuned model name here (from previous step)
FINE_TUNED_MODEL = os.getenv('CUSTOMER_CHURN_OPEN_AI_MODEL')

# Test with a sample customer profile
test_prompt = "Customer bought 500ml regularly for 2 months. Price increased by 15% recently. They raised 2 complaint(s) about product defect. Last purchase was 40 days ago. Competitor has reduced their prices recently. Predict churn risk."

print("🧪 Testing fine-tuned model...")
print(f"\n📝 Test Prompt:")
print(f"   {test_prompt}")
print(f"\n🤖 Model Response:")

try:
    response = client.chat.completions.create(
        model=FINE_TUNED_MODEL,
        messages=[
            {"role": "user", "content": test_prompt}
        ],
        temperature=0.7,
        max_tokens=200
    )
    
    print(response.choices[0].message.content)
    
except NameError:
    print("⚠️  Please set FINE_TUNED_MODEL variable first (from step 6)")
except Exception as e:
    print(f"❌ Error: {e}")


## 8. Compare with Base Model


In [ ]:
# Compare fine-tuned model with base GPT-3.5 Turbo
test_prompt = "Customer bought 500ml regularly for 2 months. Price increased by 15% recently. They raised 2 complaint(s) about product defect. Last purchase was 40 days ago. Competitor has reduced their prices recently. Predict churn risk."

print("📊 Comparing Fine-tuned vs Base Model\n")
print("=" * 60)

# Base model
print("\n1️⃣  Base GPT-3.5 Turbo:")
try:
    base_response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": test_prompt}
        ],
        temperature=0.7,
        max_tokens=200
    )
    print(base_response.choices[0].message.content)
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "=" * 60)

# Fine-tuned model
print("\n2️⃣  Fine-tuned Model:")
try:
    fine_tuned_response = client.chat.completions.create(
        model=FINE_TUNED_MODEL,
        messages=[
            {"role": "user", "content": test_prompt}
        ],
        temperature=0.7,
        max_tokens=200
    )
    print(fine_tuned_response.choices[0].message.content)
except NameError:
    print("⚠️  Please set FINE_TUNED_MODEL variable first")
except Exception as e:
    print(f"❌ Error: {e}")


## 9. Batch Inference Function


In [ ]:
def predict_churn(customer_profile: str, model_name: str = None) -> dict:
    """
    Predict churn risk for a customer profile.
    
    Args:
        customer_profile: Customer description string
        model_name: Fine-tuned model name (defaults to FINE_TUNED_MODEL)
    
    Returns:
        dict with prediction results
    """
    if model_name is None:
        model_name = FINE_TUNED_MODEL
    
    prompt = f"{customer_profile} Predict churn risk."
    
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=200
        )
        
        return {
            "success": True,
            "prediction": response.choices[0].message.content,
            "model": model_name,
            "usage": {
                "prompt_tokens": response.usage.prompt_tokens,
                "completion_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens
            }
        }
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }

# Example usage
test_customers = [
    "Customer bought 1000ml regularly for 7 months. Price increased by 3% recently. No complaints have been raised. Last purchase was 5 days ago.",
    "Customer bought 250ml slightly irregularly for 4 months. Price increased by 10% recently. They raised 1 complaint(s) about delivery delay. Last purchase was 25 days ago.",
    "Customer bought 500ml irregularly for 1 month. Price increased by 18% recently. They raised 3 complaint(s) about multiple issues. Last purchase was 42 days ago. Competitor has reduced their prices recently."
]

print("🔮 Batch Churn Predictions\n")
print("=" * 80)

for i, customer in enumerate(test_customers, 1):
    print(f"\n📋 Customer {i}:")
    print(f"   {customer}")
    print(f"\n   Prediction:")
    
    try:
        result = predict_churn(customer)
        if result["success"]:
            print(f"   {result['prediction']}")
            print(f"   (Tokens: {result['usage']['total_tokens']})")
        else:
            print(f"   ❌ Error: {result['error']}")
    except NameError:
        print("   ⚠️  Please set FINE_TUNED_MODEL variable first")
        break
    
    print("\n" + "-" * 80)


## 10. List All Fine-tuning Jobs


In [ ]:
# List all your fine-tuning jobs
print("📋 Listing all fine-tuning jobs...\n")

jobs = client.fine_tuning.jobs.list(limit=10)

if jobs.data:
    for job in jobs.data:
        print(f"Job ID: {job.id}")
        print(f"  Status: {job.status}")
        print(f"  Model: {job.model}")
        if hasattr(job, 'fine_tuned_model') and job.fine_tuned_model:
            print(f"  Fine-tuned Model: {job.fine_tuned_model}")
        print(f"  Created: {datetime.fromtimestamp(job.created_at)}")
        print()
else:
    print("No fine-tuning jobs found.")


## 11. Retrieve Training Metrics (if available)


In [ ]:
# Get training metrics for a completed job
try:
    job = check_fine_tuning_status(FINE_TUNING_JOB_ID)
    
    if job.status == 'succeeded':
        print("📊 Training Metrics:")
        print(f"   Job ID: {job.id}")
        print(f"   Model: {job.model}")
        print(f"   Fine-tuned Model: {job.fine_tuned_model}")
        
        if hasattr(job, 'trained_tokens') and job.trained_tokens is not None:
            print(f"   Trained Tokens: {job.trained_tokens:,}")
        
        # Retrieve training events
        try:
            events = client.fine_tuning.jobs.list_events(job.id, limit=50)
            
            if events.data:
                print(f"\n   Training Events:")
                for event in events.data:
                    print(f"     [{datetime.fromtimestamp(event.created_at)}] {event.message}")
            else:
                print("\n   No training events available.")
        except Exception as e:
            print(f"\n   Could not retrieve training events: {e}")
    else:
        print(f"⏳ Job not yet completed. Current status: {job.status}")
except NameError:
    print("⚠️  Please set FINE_TUNING_JOB_ID variable first")


## 12. Cost Estimation


In [ ]:
# Estimate fine-tuning costs
# GPT-3.5 Turbo fine-tuning pricing (as of 2024):
# Training: $8.00 per 1M tokens
# Usage: $3.00 per 1M input tokens, $6.00 per 1M output tokens

TRAINING_COST_PER_MILLION_TOKENS = 8.00
INFERENCE_INPUT_COST_PER_MILLION = 3.00
INFERENCE_OUTPUT_COST_PER_MILLION = 6.00

try:
    job = check_fine_tuning_status(FINE_TUNING_JOB_ID)
    
    if hasattr(job, 'trained_tokens') and job.trained_tokens is not None:
        trained_tokens = job.trained_tokens
        training_cost = (trained_tokens / 1_000_000) * TRAINING_COST_PER_MILLION_TOKENS
        
        print("💰 Cost Estimation:")
        print(f"   Trained Tokens: {trained_tokens:,}")
        print(f"   Training Cost: ${training_cost:.2f}")
        print(f"\n   Note: Additional costs apply for inference:")
        print(f"   - Input: ${INFERENCE_INPUT_COST_PER_MILLION} per 1M tokens")
        print(f"   - Output: ${INFERENCE_OUTPUT_COST_PER_MILLION} per 1M tokens")
    else:
        print("⏳ Training not yet complete. Cost estimation will be available after training.")
except NameError:
    print("⚠️  Please set FINE_TUNING_JOB_ID variable first")
except Exception as e:
    print(f"❌ Error: {e}")


## Notes and Tips

1. **Training Time**: Fine-tuning typically takes 10 minutes to several hours depending on dataset size
2. **Model Availability**: The fine-tuned model will be available immediately after training completes
3. **Costs**: Monitor your usage in the OpenAI dashboard
4. **Best Practices**: 
   - Use validation data to evaluate model performance
   - Start with fewer epochs and increase if needed
   - Monitor for overfitting
5. **API Limits**: Check OpenAI's rate limits for fine-tuning jobs

## Resources
- [OpenAI Fine-tuning Documentation](https://platform.openai.com/docs/guides/fine-tuning)
- [OpenAI Pricing](https://openai.com/pricing)
- [Fine-tuning Best Practices](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset)
